In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_02_train.py"
CELLS_SHA = "7b10d92c26bcd122"

# 02 - Train the joint counter + separator

**Accelerator: GPU T4 x2** (or P100). **Runtime: up to 11 h per session, resumable.**

This is where the quota goes. Read the resume section before you start it.

## How a 12-hour cap is survived

Kaggle gives you 12 hours and then takes the machine away, and `/kaggle/working` is wiped
between sessions unless you **Save Version**. Three mechanisms handle that:

| Failure | Response |
|---|---|
| Time budget expires | `TimeBudget` stops **cleanly and exits 0** at 11 h, so Save Version still captures the checkpoint |
| Hard kill (tab closed, OOM) | mid-epoch checkpoint every 400 steps - you lose minutes |
| New session | `find_resume` searches `/kaggle/working`, then every attached input dataset, for the newest `last.pt` |

**The resume loop, in full:**

1. Run this notebook. It stops on its budget and prints a RESUME banner.
2. **Save Version -> Save & Run All (Commit).** Wait for it to finish.
3. **+ Add Input -> Notebook Output ->** *this* notebook's latest version.
4. Re-run. It picks up `last.pt` by itself. No path edits, ever.

Optimizer moments, scheduler position, AMP scaler and RNG streams all resume exactly.

## Before you run

1. **Settings -> Accelerator -> GPU T4 x2.**
2. **+ Add Input -> Notebook Output ->** `00_build_dataset`.
3. From the second session onward, **also** add this notebook's own previous output.

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate.git"
REPO_DIR = "/kaggle/working/speaker-count-separate"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import csnet  # noqa: E402

print("csnet", csnet.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_store, find_recipes
import glob

STORE = autodetect_store()
# Not a raw glob: a training notebook's output carries a whole git clone, so
# `glob("/kaggle/input/**/recipes_dev.csv")[0]` can return the copy committed to
# the repo instead of the one notebook 00 built -- decided by filesystem walk
# order, which changes with whatever inputs happen to be attached. Two notebooks
# then evaluate one checkpoint against two different frozen sets and disagree.
RECIPES_DEV = find_recipes("recipes_dev.csv", STORE)

print("store      :", STORE)
print("dev recipes:", RECIPES_DEV)
assert STORE and RECIPES_DEV, "attach the 00_build_dataset notebook output via '+ Add Input'"

previous = sorted(glob.glob("/kaggle/input/**/last.pt", recursive=True))
print("\ncheckpoints visible from a previous session:", previous or "none (first session)")

## Choose the run

| Config | Params | Speed | Use for |
|---|---|---|---|
| `configs/paper.yaml` | 5.3 M | 1x | the headline result |
| `configs/small.yaml` | 1.9 M | ~5x faster | ablations, and if quota is tight |
| `configs/tiny.yaml` | 0.3 M | CPU-able | plumbing only, learns nothing |

**Set `EPOCHS` from the measurement, not from ambition.** A timed run on GPU T4 x2 at
batch 12 gave **1.72 s/step**, so 1000 steps plus validation is about **30 minutes an
epoch** and an 11-hour session buys roughly **21 epochs**. The free weekly quota is 30
GPU-hours, so:

| EPOCHS | sessions | GPU-h | leaves for evaluation |
|---|---|---|---|
| 40 | 2 | ~22 | 8 h |
| 60 | 3 | ~31 | nothing -- over the weekly quota |

40 is the default here for that reason. State it in the report as a budget decision. Our
numbers will sit a decibel or two below published baselines; that is expected and
defensible. Half a joint model plus half an interpretability study is not.

## Memory: it was DataParallel

A session died with `exit -9` -- the Linux OOM killer, which takes the process with no
traceback. Telemetry showed the container limit is 30 GiB and host memory then climbed
about 8 MiB **every step**, dead linear, until it hit the ceiling four hours in. 8 MiB is
one batch: mix 1.15 + refs 5.76 + noise 1.15 MB.

`scripts/10_memory_bisect.py` settled it in one 13-minute run by measuring the memory
slope of the real training loop with one component switched off at a time:

| configuration | cgroup MiB/step | rss MiB/step | ms/step |
|---|---|---|---|
| baseline | 17.34 | 7.97 | 1750 |
| workers 0 | 8.12 | 12.74 | 1792 |
| **no dataparallel** | **0.13** | **0.00** | **1140** |
| workers 0 + no dataparallel | 0.04 | 4.66 | 1177 |

Both flat rows have DataParallel off; both leaking rows have it on. So
`train.dataparallel` is now **false** in `configs/base.yaml`, and that is a win three
times over:

* the leak is gone -- 0.1 MiB/step instead of 17;
* it is **35 % faster** -- 1140 ms/step against 1750. Replicating a 5.3 M-parameter
  model across two GPUs every forward and gathering `(B, 6, 24000)` outputs back costs
  more than the second T4 returns on a model this small;
* the `no amp` configuration crashed twice with `CUDA error: misaligned address`, in a
  fresh process each time, and only ever with DataParallel on.

**The second T4 is now idle.** That is the right trade at this model size, and the
measurement above is the justification to put in the report.

## Budget, re-measured

At **1045 ms/step**, 1000 steps plus validation is about **19 minutes an epoch**, so an
11-hour session buys roughly **37 epochs**. 40 epochs fits in two sessions comfortably.

## The counting head collapsed, and what was done about it

After 38 epochs, separation was working -- validation SI-SDR went from **-11.35 to
+0.50 dB** -- and the counting head had not moved at all: cross-entropy pinned at 1.611
where ln(5) is 1.6094, accuracy at 0.200 where chance is 0.200, unchanged since step 2900.

`scripts/11_inspect_count_head.py` read the answer straight off the checkpoint:

| measurement | value |
|---|---|
| fc1 units never active | **128 of 128** |
| post-ReLU values that are zero | **100 %** |
| logit variation across samples | **0.0000** |
| pooled feature variation | 0.4144 |

Every hidden unit was negative for every input, so ReLU zeroed the layer and `fc2` could
only emit its bias -- which settles on the class marginal, uniform for a balanced set,
which *is* ln(5). The features feeding it varied perfectly well, so nothing upstream was
wrong. And a dead ReLU receives no gradient, so it could never have recovered: all 96
test samples were predicted `N=3`, the largest element of the bias vector.

The head now has **LayerNorm on the pooled statistics, LayerNorm after `fc1`, and GELU**
instead of ReLU. The LayerNorm after `fc1` subtracts the mean across the hidden
dimension, so a uniform negative shift -- which is what one large early step produces,
and the early steps are large because the loss starts near 64 -- is removed rather than
saturating anything. A test drives the new head into the exact state that killed the old
one and asserts it still varies and still receives gradient.

Two things also changed so this cannot cost thirteen hours again:

* the trainer **warns** if counting accuracy sits at chance for four epochs, naming the
  inspector to run;
* `--reset_count_head` re-initialises the head on resume while keeping the separator,
  because a collapsed head's weights are a local optimum with no gradient out of them.

## Recovering from here

The 13.5 hours of separator training are worth keeping; only the head needs redoing.
Set `RECOVER = True` in the cell below: it resumes from your checkpoint, re-initialises
the head, freezes the separator, and trains the counting head alone -- Gate 6 of the plan.
With the separator frozen a step takes **331 ms instead of 1045**, so 12 epochs is about
**70 minutes**, not eleven hours.

Note `--extra_epochs 12` rather than `--set train.epochs=12`. `train.epochs` is an
absolute target: resuming at epoch 38 and asking for 12 gives `range(38, 12)`, which is
empty, so the run trains nothing and exits 0. The trainer now refuses that instead of
pretending it worked.

## Gate 2 - the control that was never run

The full evaluation came back with the counter working and the separator not:

| | ours | floor | published |
|---|---|---|---|
| counting accuracy | **44.9 %** | 35.0 % naive, 20 % chance | - |
| SI-SDRi, N=2..5 | **+1.2 dB** | 0 dB (the mixture) | 14.76 dB at N=2 |
| IRM/IBM oracle | - | - | 10.6-12.8 dB |

Before changing the architecture, it is worth knowing that it is not broken. A single
fixed batch of four two-speaker mixtures, 250 steps, nothing else changed:

| objective | SI-SDR reached |
|---|---|
| separation term alone | **28.36 dB** |
| the full production loss | **19.66 dB** |

So the model, the loss and the optimiser are all capable -- no bug. What the A/B also
shows is that the auxiliary objectives cost about **8.7 dB of progress at matched step
count**: the silence term is driven to 0.000 within a hundred steps, and separation is
nine decibels behind for all of them. It is a real tax, and it is not the whole story,
because 38,800 steps of the real run reached 0.19 dB on its own *training* data.

What was left was the thing the design document said to check first and the project never
did: **is +1.2 dB the pipeline's ceiling, or the pooled N=1..5 task's?** `GATE2 = True`
answered it in 2.53 GPU-hours -- two speakers in every mixture, no counting term, no
silence term, which is the standard fixed-N setup the 14.76 dB was measured under:

| run | epochs | GPU-h | val SI-SDR |
|---|---|---|---|
| pooled N=1..5, full loss | 38 | ~13.5 | **0.50 dB** |
| fixed N=2, separation only | 8 | 2.5 | **7.40 dB** |

**The pipeline is sound.** Fifteen times the separation in a fifth of the time, same
code, same data, same architecture. So +1.2 dB was never the machine's ceiling: it is
what one model costs when it has to serve five speaker counts and three auxiliary
objectives on a forty-epoch budget. That is the finding, and it is a real one -- the
single-batch ablation above already measured the objectives at -8.7 dB and this puts a
number on the rest of it.

## Pricing the objectives on the real task -- `SILOW`

Gate 2 answered "is the pipeline broken?" with a run that moved two things at once: it
dropped to a single speaker count *and* switched three objectives off. The fifteenfold is
real but unattributed -- the pooled task and the auxiliary objectives are both inside it,
so the report can only say "and/or", which is the weakest form a finding can take.

`SILOW = True` moves one of them on its own. `configs/silow.yaml` is `paper.yaml` with
the silence weight cut tenfold, 1.0 -> 0.1: the same pooled N=1..5 mixtures, the same
counting term, the same 38 epochs against the same dev set. Whatever it gains over
0.50 dB is the silence term's price on the real task, measured rather than extrapolated
from four memorised mixtures.

Why that term and not one of the others: the silence penalty is not scale-invariant and
SI-SDR is, so shrinking every slot at once drives the penalty to zero while leaving
separation untouched. `14_objective_ablation.py` watched it reach 0.000 inside a hundred
steps with separation nine decibels behind. It is the term with a free descent direction,
which makes it the one worth a run.

| | epochs | GPU-h | val SI-SDR |
|---|---|---|---|
| pooled, `w_sil` 1.0 -- the reference | 38 | ~13.5 | 0.50 dB |
| pooled, `w_sil` 0.1 -- this run | 38 | ~12 | ? |
| fixed N=2, no silence term at all | 8 | 2.5 | 7.40 dB |

**Two sessions**, 11 h then about 1.2 h, then notebook 04 with `ONLY = "ckpt_silow"`.
Roughly 12.5 GPU-h of the weekly 30. The second session is what buys matched epochs, and
matched epochs are the comparison: `--set train.epochs=35` fits one session, but then the
result has to be reported as 35 against 38, which is a weaker claim than it looks.

**It will not fix counting.** The count head reads pooled statistics that this knob does
not touch, and in fp32 it answers "1 speaker" to almost every mixture. If the counting
row moves, suspect the measurement before believing it.

Two things in that run's log are expected, not faults. `w_count` is zero, so the
counting head is untrained by design and its accuracy is noise -- the collapse warning
no longer fires when counting is switched off. And `lr 0.00e+00` at the end is the
cosine schedule arriving at zero on the last step, which is what it is for.

In [ ]:
CONFIG = "configs/paper.yaml"
EPOCHS = 40                # measured: ~19 min/epoch, so ~37 epochs per 11 h session
BATCH_SIZE = 12          # drop to 8 if you hit CUDA OOM
TIME_BUDGET_H = 11.0     # stop cleanly before Kaggle's 12 h cap
STEPS_PER_EPOCH = 1000

# The switches that change what this notebook does. They live here, above everything that
# reads them, because CKPT_DIR depends on them: each variant writes to its own directory,
# and a progress plot pointed at another one silently shows nothing. Set at most one.
RECOVER = False          # True: keep the separator, re-init the counting head, train it alone
GATE2 = False            # True: the fixed-N=2 control run, ~2.5 h -- see "Gate 2" above
SILOW = False            # True: pooled rerun at w_sil 0.1, ~12 h -- see "Pricing" above
BISECT = False           # True: spend ~15 min finding which component leaks, and train nothing
EXTRA_EPOCHS = 12        # RECOVER only: this many MORE epochs, counted from the checkpoint

assert sum([RECOVER, GATE2, SILOW, BISECT]) <= 1, "these are alternatives, not a stack"

CKPT_DIR = ("/kaggle/working/ckpt_gate2" if GATE2 else
            "/kaggle/working/ckpt_silow" if SILOW else
            "/kaggle/working/ckpt_count" if RECOVER else
            "/kaggle/working/ckpt")
RUN_CONFIG = ("configs/gate2.yaml" if GATE2 else
              "configs/silow.yaml" if SILOW else CONFIG)

# Which attached run this one may resume from. GATE2 and SILOW are separate experiments
# that begin at step 0, so their session-two resume has to be pinned to their own
# directory: find_resume ranks by global step, and the pooled reference at 50,800 steps
# beats anything either of them will reach. Unpinned, session two of SILOW would continue
# the reference model under a loss its weights were never grown with -- eleven hours spent
# making the one comparison this run exists to avoid. RECOVER stays unpinned on purpose:
# resuming *from* the pooled run into a fresh directory is what it is for.
RESUME_CONTAINS = os.path.basename(CKPT_DIR) if (GATE2 or SILOW) else ""

print("writing to :", CKPT_DIR)
print("config     :", RUN_CONFIG)
print("resume from:", f"attached paths containing {RESUME_CONTAINS!r}"
      if RESUME_CONTAINS else "any attached checkpoint (furthest along wins)")

## Preflight (30 seconds, no quota)

Every expensive failure this project has had was visible in the first minute: cells
imported two pushes ago, a resume that picked the five-step dry run, an epoch range that
was empty, a container already half full of RAM. This cell checks those and **stops the
notebook** rather than letting an eleven-hour session discover them.

If it says `PREFLIGHT: STOP`, read the `->` line under the failing row. Nothing has been
spent yet.

In [ ]:
run(f"python scripts/12_preflight.py --for {'recover' if RECOVER else 'train'}"
    f" --config {RUN_CONFIG} --store {STORE}"
    f" --recipes_dev {RECIPES_DEV}"
    f" --ckpt_dir {CKPT_DIR} --time_budget_h {TIME_BUDGET_H}"
    # GATE2 and SILOW carry their epoch count in the config, so passing --epochs here would
    # override it with the pooled default and check an epoch range nothing is going to run.
    + (f" --extra_epochs {EXTRA_EPOCHS}" if RECOVER else
       "" if GATE2 or SILOW else f" --epochs {EPOCHS}")
    + (f" --resume_contains {RESUME_CONTAINS}" if RESUME_CONTAINS else "")
    + f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}"
    f" --set train.batch_size={BATCH_SIZE} train.steps_per_epoch={STEPS_PER_EPOCH}")

## Plumbing check first (30 seconds)

Five steps and one eval batch. Never spend an hour of quota discovering that a path is wrong.

In [ ]:
run(f"python scripts/04_train.py --config {RUN_CONFIG}"
    f" --store {STORE} --recipes_dev {RECIPES_DEV}"
    f" --ckpt_dir /kaggle/working/_dryrun --resume none --dry_run")

## Train

Before it commits, the script times 20 real steps and prints how many epochs the remaining
budget actually buys. **That measurement replaces the FLOP table** - depthwise separable
1-D convolutions are memory-bound, so a peak-TFLOPS estimate is not worth much.

`--resume auto` means: use `CKPT_DIR/last.pt` if it exists, otherwise the **furthest-along**
`last.pt` in any attached dataset, otherwise start fresh. Furthest-along, not newest: a
notebook output holds `_dryrun/last.pt` beside the real one and picking by timestamp is a
coin flip that, lost, restarts eleven hours of training from step 5.

In [ ]:
if BISECT:
    run(f"python scripts/10_memory_bisect.py --store {STORE}"
        f" --config {CONFIG} --set train.batch_size={BATCH_SIZE}")
elif GATE2:
    # The control the design document asks for and the project skipped. Two speakers in
    # every mixture, no counting term, no silence term: the standard fixed-N setup that
    # the published 14.76 dB was measured under. `neg_loss` is the right metric to track
    # here because with w_count and w_sil at zero the loss *is* negative SI-SDR.
    run(f"python scripts/04_train.py"
        f" --config configs/gate2.yaml"
        f" --store {STORE}"
        f" --recipes_dev {RECIPES_DEV}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --dev_n 2"
        f" --resume auto --resume_contains {RESUME_CONTAINS}"
        f" --time_budget_h {TIME_BUDGET_H}"
        f" --best_metric neg_loss"
        f" --set train.batch_size={BATCH_SIZE}"
        f" train.steps_per_epoch={STEPS_PER_EPOCH}")
elif SILOW:
    # One knob away from the pooled reference: the silence weight, 1.0 -> 0.1. Same five
    # speaker counts, same counting term, same 38 epochs, so the gap in val SI-SDR is the
    # silence term's price on the real task rather than on one memorised batch. `p_si_snr`
    # is the metric the reference was scored on; read `val_sisdr` in the table below for
    # the comparison itself, because P-SI-SNR also carries the counting head's failures.
    run(f"python scripts/04_train.py"
        f" --config configs/silow.yaml"
        f" --store {STORE}"
        f" --recipes_dev {RECIPES_DEV}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --resume auto --resume_contains {RESUME_CONTAINS}"
        f" --time_budget_h {TIME_BUDGET_H}"
        f" --best_metric p_si_snr"
        f" --set train.batch_size={BATCH_SIZE}"
        f" train.steps_per_epoch={STEPS_PER_EPOCH}")
elif RECOVER:
    # Gate 6: the separator is frozen, so only the counting head learns. Its loss is the
    # only one left, which is also the cleanest test of whether the features can support
    # counting at all.
    run(f"python scripts/04_train.py"
        f" --config {CONFIG}"
        f" --store {STORE}"
        f" --recipes_dev {RECIPES_DEV}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --resume auto --reset_count_head"
        f" --time_budget_h {TIME_BUDGET_H}"
        f" --best_metric count_acc"
        f" --extra_epochs {EXTRA_EPOCHS}"   # more epochs from here, not an absolute target
        f" --set train.batch_size={BATCH_SIZE}"
        f" train.steps_per_epoch={STEPS_PER_EPOCH}"
        f" train.freeze_separator=True"
        f" loss.w_sep=0.0 loss.w_sil=0.0 loss.w_noise=0.0")
else:
    run(f"python scripts/04_train.py"
        f" --config {CONFIG}"
        f" --store {STORE}"
        f" --recipes_dev {RECIPES_DEV}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --resume auto"
        f" --time_budget_h {TIME_BUDGET_H}"
        f" --best_metric p_si_snr"
        f" --set train.epochs={EPOCHS}"
        f" train.batch_size={BATCH_SIZE}"
        f" train.steps_per_epoch={STEPS_PER_EPOCH}")
if BISECT:
    print("BISECT is True -- skipping training. Set it back to False once the table")
    print("above names the component to switch off.")

## Progress

In [ ]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

log = os.path.join(CKPT_DIR, "train_log.csv")
df = pd.read_csv(log) if os.path.exists(log) else None
if df is not None and df.empty:
    # A session that stopped before finishing an epoch writes a header and no rows.
    # Plotting that raises, which turns a clean stop into a failed notebook.
    print("train_log.csv has no completed epochs yet -- this session stopped early.")
    print("The checkpoint is still saved; the curves appear once an epoch finishes.")
    df = None
if df is not None:
    display(df.tail(12))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
    axes[0].plot(df["epoch"], df["train_loss"], label="train")
    axes[0].plot(df["epoch"], df["val_loss"], label="val")
    axes[0].set_ylabel("loss"); axes[0].legend()
    axes[1].plot(df["epoch"], df["train_sisdr"], label="train")
    axes[1].plot(df["epoch"], df["val_sisdr"], label="val")
    axes[1].set_ylabel("matched SI-SDR (dB)"); axes[1].legend()
    axes[2].plot(df["epoch"], df["val_count_acc"] * 100, label="count accuracy %")
    ax2 = axes[2].twinx(); ax2.grid(False)
    ax2.plot(df["epoch"], df["val_p_si_snr"], color="tab:red", label="P-SI-SNR")
    axes[2].set_ylabel("count accuracy (%)"); ax2.set_ylabel("P-SI-SNR (dB)")
    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(f"total wall clock across all sessions: {df['wall_h'].max():.2f} h")
    columns = [("val_p_si_snr", "P-SI-SNR", "dB", 1.0), ("val_sisdr", "SI-SDR", "dB", 1.0)]
    if not GATE2:
        columns.append(("val_count_acc", "count accuracy", "%", 100.0))
    for column, label, unit, scale in columns:
        best = df[column].dropna() if column in df else df.get(column)
        if best is None or best.empty:
            print(f"no validated epoch yet, so there is no best {label} to report")
        else:
            print(f"best val {label}: {scale * best.max():.2f} {unit} "
                  f"at epoch {int(df.loc[best.idxmax(), 'epoch'])}")
    if GATE2:
        # This run sets w_count to zero, so the head is untrained by design and its
        # accuracy is noise. Printing a floor for it would invite a false alarm.
        print("counting is not trained in a GATE2 run -- ignore the count columns")
    else:
        # The bar is not chance (20 %) but the naive predictor. 35.0 % is the figure
        # measured on the *test* set with 5-fold CV in notebook 04; the 41.1 % quoted
        # earlier was dev with 2-fold, a different measurement rather than a better one.
        print("naive-predictor floor to beat: 35.0 % on test   (chance is 20.0 %)")
elif not os.path.exists(log):
    print("no train_log.csv yet")

In [ ]:
for name in ("last.pt", "best.pt"):
    path = os.path.join(CKPT_DIR, name)
    if os.path.exists(path):
        print(f"{name:<8} {os.path.getsize(path) / 1e6:6.1f} MB")

## If it stopped on the budget

It printed a `======== RESUME ========` block. That is the normal, healthy ending - not an
error. Do exactly this:

1. **Save Version -> Save & Run All (Commit).**
2. When it finishes: **+ Add Input -> Notebook Output ->** this notebook, latest version.
3. Re-run. It continues from the exact step it stopped at.

Repeat until `EPOCHS` is reached. Each session costs one Save Version, nothing else.

---

### Gate 2 - do this before trusting anything at N > 2

Reproduce fixed-N=2 within about 1 dB of the published **14.76 dB** SI-SDRi on the official
Libri2Mix test set. Notebook `04_evaluate` does it with `--libri2mix_dir`. **If that fails,
stop and fix it** - every downstream number is uninterpretable until it passes.

### Gate 6 - the count head alone

If joint training struggles, freeze the separator and train only the count head (~2 h):

```
--set train.freeze_separator=True loss.w_sep=0.0 loss.w_sil=0.0 loss.w_noise=0.0
```

If that cannot beat ~85 % with the leak mitigated, fall back to fixed-N=3 plus the
interpretability work and say so in the report.